In [10]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from datetime import date, datetime
import requests
import os
import re 
import json
from contextlib import redirect_stderr
import pdfplumber

In [4]:
driver = webdriver.Firefox()
driver.get("https://www.pwc.in/research-insights.html")
driver.implicitly_wait(7) #to let the site load 

# Wait and click the cookie consent button if present
try:
    cookie_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
    )
    cookie_button.click()
except:
    print("No cookie banner")

#find button
button = driver.find_element(By.XPATH, '//button[text()="Load more"]')
i=0
while i<5:
    button.click()
    time.sleep(2)
    i+=1

In [5]:
# Function to download and extract PDF content
import logging

# Suppress pdfminer logs
logging.getLogger("pdfminer").setLevel(logging.ERROR)

# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def extract_pdf_content(pdf_url):
    try:
        # Download the PDF
        response = requests.get(pdf_url)
        with open("temp.pdf", "wb") as f:
            f.write(response.content)

        # Suppress warnings by redirecting stderr
        with open(os.devnull, "w") as devnull, redirect_stderr(devnull):
            with pdfplumber.open("temp.pdf") as pdf:
                pdf_content = "\n".join([page.extract_text() for page in pdf.pages])

        # Clean up the temporary file
        os.remove("temp.pdf")
        return pdf_content
    except Exception as e:
        print(f"Error extracting PDF content from {pdf_url}: {e}")
        return ""

In [6]:
def clean_content(text):
    # Step 1: Remove unwanted characters
    cleaned_text = re.sub(r'[\r\n]+', ' ', text) #Remove \n characters
    cleaned_text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII characters
    cleaned_text = re.sub(r'\u2013', '-', cleaned_text)   # Replace en-dash with hyphen
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)     # Collapse multiple spaces

    # Step 2: Remove page numbers and headers/footers
    cleaned_text = re.sub(r'\d+ \| PwC \| ', '', cleaned_text)
    cleaned_text = re.sub(r'Table of contents', '', cleaned_text)
    # Step 3: Remove extra whitespace and normalize formatting
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text

In [7]:
all_articles = driver.find_elements(By.TAG_NAME, 'article')

extracted_urls = []
start = time.time()
for article in all_articles:
    date_element = article.find_element(By.TAG_NAME, 'p')
    date_string = date_element.text
    article_date = datetime.strptime(date_string, "%d/%m/%y").date()
    if article_date.month == 3 and article_date.year == 2025:
        link = article.find_element(By.TAG_NAME, 'a').get_attribute('href')
        if link.startswith("https://www.pwc.in/ghost-templates/"):
            title = link.split("ghost-templates/")[1].split(".html")[0].replace("-", " ")
            content = extract_pdf_content(link)
            content_cleaned = clean_content(content)
            extracted_urls.append({
                "title": title,
                "date": str(article_date),
                "content": content_cleaned,
                "url": link
            })
end = time.time()

In [8]:
extracted_urls

[{'title': 'agentic ai in the human capital management hcm industry',
  'date': '2025-03-27',
  'content': 'Agentic AI in the human capital management (HCM) industry Leverage Oracles agentic AI technology to strengthen your HR function AI agents assisting Chief Human Resources Officers (CHROs) 01 02 03 04 Recruiting an AI agent Employee lifecycle Performance Compliance and management agent management AI agent lifecycle AI agent Automates job postings, generates candidate summaries and provides Assists with onboarding and Suggests goals, drafts feedback and Assists with onboarding, internal hiring insights internal mobility summarises performance reviews mobility and regulatory adherence Use cases: Uses generative AI to quickly generate personalised job descriptions that align with the company culture Relies on candidate score AI to compare applicants based on skills and job history Uses career planning guide AI to help employees see growth opportunities within the company PwC Agentic A

In [14]:
try:
    with open('extracted_urls.json', 'w') as f:
        json.dump(extracted_urls, f, indent=4)
    print("Data saved to extracted_urls.json")
except Exception as e:
    print(f"Error saving to JSON file: {e}")

Data saved to extracted_urls.json


In [12]:
duration = end-start
print(duration)

92.07696628570557
